In [12]:
import sys
import os
sys.path.append("..")

In [13]:
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.neural_network import MLPClassifier
from bonXAI.core.preprocessing import Preprocessor
from bonXAI.core.explainer import Explainer
from bonXAI.core.evaluation import Evaluator
from bonXAI.core.utils import set_global_seed, possible_g_values, possible_num_bins_values

from openxai.model import LoadModel, ReturnLoaders

# pip install stein-thinning https://zoltansz.github.io/utils/DSS/slides/2022_05_30_Lester_Mackey_slides.pdf http://stein-thinning.org
from stein_thinning.thinning import thin

SEED = 42
set_global_seed(SEED)

In [14]:
# generate data
X, y = make_classification(n_samples=1000, n_features=7, n_classes=2, random_state=SEED)
df_data = pd.DataFrame(X, columns=[f"feature_{i}" for i in range(X.shape[1])])
df_data["label"] = y

In [15]:
# train model
model = MLPClassifier(hidden_layer_sizes=(10,), max_iter=500, random_state=SEED)
model.fit(X, y)

MLPClassifier(hidden_layer_sizes=(10,), max_iter=500, random_state=42)

In [16]:
# calculate Ground Truth
gt_explainer = Explainer(model=model, name="shap", variant="kernel", seed=SEED)
exp_gt, t_gt = gt_explainer.explain(X, y)
exp_gt_mean = exp_gt.mean(axis=0)

Using 1000 background data samples could cause slower run times. Consider using shap.sample(data, K) or shap.kmeans(data, K) to summarize the background as K samples.


### Kernel thinning

In [17]:
pre = Preprocessor(method="compress", model=model, kernel="gaussian", seed=SEED)
X_red, y_red, idx, t_comp = pre.run(X, y)

In [18]:
X_red.shape

(16, 7)

### Stein Thinnings KST

In [19]:
# Fit a Gaussian to X and compute score (grad log p)
mu = X.mean(axis=0)
Sigma = np.cov(X, rowvar=False) + 1e-6*np.eye(X.shape[1])  # tiny ridge for stability
prec = np.linalg.inv(Sigma)
grad = -(X - mu) @ prec     # shape: (n, d)

In [20]:
m = 16  # how many points to keep
idx = thin(X, grad, m)      # indices of selected rows  :contentReference[oaicite:1]{index=1}

X_sel = X[idx]
y_sel = y[idx]

In [22]:
X_sel.shape

(16, 7)

### Compare

In [23]:
# KST
evaluator = Evaluator(ground_truth_explanation=exp_gt_mean, reference_points=X)
explainer = Explainer(model=model, name="shap", variant="kernel", seed=SEED)
values, t_exp = explainer.explain(X_sel, y_sel)
exp_mean = values
if exp_mean.ndim > 1:
    exp_mean = values.mean(axis=0)
row = evaluator.evaluate_explanation(exp_mean, t_exp, len(X_sel))
row

{'mae': 1.3091927095992033e-17,
 'top_k': 0.6,
 'time': 0.14484405517578125,
 'size': 16}

In [24]:
# KT
evaluator = Evaluator(ground_truth_explanation=exp_gt_mean, reference_points=X)
explainer = Explainer(model=model, name="shap", variant="kernel", seed=SEED)
values, t_exp = explainer.explain(X_red, y_red)
exp_mean = values
if exp_mean.ndim > 1:
    exp_mean = values.mean(axis=0)
row = evaluator.evaluate_explanation(exp_mean, t_exp, len(X_red))
row

{'mae': 1.446849212858949e-17,
 'top_k': 0.8,
 'time': 0.14229106903076172,
 'size': 16}

KST doesn't seem to be worse then KT
